# 第3回 両替予算の計算


第2回の続きで、2024年のドル・円レート（日銀、東京17時）を使う。値の比較、`if` による条件分岐、リストと繰り返し、折れ線グラフ、平均・最小・最大を順に練習し、演習1では12万円で買えるドルを計算する。


## 実行の準備

`finance_data.zip` をNotebookと同じフォルダに置き、下のセルを実行する。セルは上から順に実行すること。「変数が見つからない」というエラーが出たら、前のセルを実行し忘れている。

In [ ]:
from pathlib import Path
from zipfile import ZipFile

if not Path("course_data.py").exists():
    with ZipFile("finance_data.zip") as archive:
        archive.extractall(".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course_data import (
    load_boj_fx, load_boj_monthly, load_fed_fx, load_fed_rates,
    configure_plotting, format_axis,
)

configure_plotting()
pd.options.display.float_format = "{:.3f}".format


## 第2回で作った変数

新しいNotebookには第2回の変数が残っていないので、次のセルで作り直す。`fx` は日銀のドル・円の表、`rate` はその列から欠測の日を除いたSeries、`rate_2024` は2024年の部分である。`first` と `last` は2024年の最初と最後の観測、`change` はその差（円／ドル）を表す。

円の予算 $B$ で買えるドルの量は、$B$ をレート $S$（円／ドル）で割った $Q=B/S$ である。演習1でもこの式を使う。


In [ ]:
fx = load_boj_fx()
rate = fx["usd_jpy"].dropna()
rate_2024 = rate.loc["2024"]

first = rate_2024.iloc[0]
last = rate_2024.iloc[-1]
change = last - first

print(f"2024年の観測数：{len(rate_2024):,}")
print(f"最初の観測：{rate_2024.index[0].date()}、{first:.2f} 円／ドル")
print(f"最後の観測：{rate_2024.index[-1].date()}、{last:.2f} 円／ドル")
print(f"変化額：{change:.2f} 円／ドル")


## データ型と比較

Pythonでは、数と文字を別の種類（データ型）として扱う。`type()` で、値がどの種類かを調べられる。


In [ ]:
label = "ドル・円"
print(type(label))
print(type(first))


`str` は文字列（文字の並び）、`numpy.float64` は小数を含む数を表す。`first` は数なので計算に使えるが、文字列の `"ドル・円"` は割り算などの計算には使えない。


次に、2つの数を比べる。`>` や `<` で比べると、結果は `True`（正しい）か `False`（正しくない）になる。


In [ ]:
print(last > first)
print(last < first)


最後のレート（157.89円）は最初のレート（143.38円）より大きいので、`last > first` は `True`、`last < first` は `False` になった。


`if` を使うと、条件によって実行する処理を変えられる。

- `if 条件:` の条件が `True` なら、その下の字下げした行を実行する。
- `elif 条件:` は、上の条件が `False` だったときに、次の条件を調べる。
- `else:` は、どの条件にも当てはまらなかったときに実行する。

行末の `:` と、次の行の字下げ（先頭の空白）が、どこまでがその処理かを示す。


In [ ]:
if change > 0:
    print("2024年の最初と最後の観測を比べると、円安・ドル高です。")
elif change < 0:
    print("2024年の最初と最後の観測を比べると、円高・ドル安です。")
else:
    print("2024年の最初と最後の観測値は同じです。")


`change` は14.51でプラスなので、最初の `print` だけが実行された。

値を変えると、実行される行も変わる。下のセルでは、変化額が −3 円だった場合を `test_change` という別の名前で試す。


In [ ]:
test_change = -3

if test_change > 0:
    print("円安・ドル高です。")
elif test_change < 0:
    print("円高・ドル安です。")
else:
    print("変化はありません。")


今度は2つ目の `print` が実行された。`-3` を `0` に書き換えて実行し、`else` の行が実行されることも確かめてみよう。


## リストと繰り返し

いくつかの値を順に並べたものをリストという。まず2024年のレートから、先頭の3件を取り出す。`.iloc[:3]` は「先頭から3件」を表す。


In [ ]:
rate_2024.iloc[:3]


日付とレートが3件並んだ。これを `.tolist()` でリストに変え、`values` という名前を付ける。


In [ ]:
values = rate_2024.iloc[:3].tolist()
values


`[ ]` の中に、3つのレートがカンマで区切られて並んだ。リストにすると日付はなくなり、値だけが残る。

リストの件数は `len()` で数え、何番目の値かは `values[番号]` で取り出す。Pythonでは先頭を0番目と数えるので、先頭の値は `values[0]`、その次の値は `values[1]` になる。


In [ ]:
print(len(values))
print(values[0])
print(values[1])


件数は3、0番目は143.38、1番目は145.07である。


`for value in values:` は、リストの値を1つずつ `value` に入れて、その下の字下げした行を繰り返し実行する。


In [ ]:
for value in values:
    print(f"1ドル = {value:.2f} 円なら、1,000ドルの支払いは {1_000 * value:,.0f} 円")


3つの値について、同じ `print` が3回実行された。リストを作るセルの `.iloc[:3]` を `.iloc[:5]` に変えて、そのセルとこのセルを実行し直してみよう。今度は5行表示される。


## レートの推移

2024年のレートを折れ線グラフにする。グラフを描くコードは、次の部分からなる。

- `fig, ax = plt.subplots(figsize=(9, 4))` で図を用意する。`fig` は図全体、`ax` は線を描く場所で、`figsize` は図の横と縦の大きさである。
- `ax.plot(横軸, 縦軸)` で線を描く。横軸は日付（`rate_2024.index`）、縦軸はレート（`rate_2024`）で、`linewidth=1` は線の太さである。
- `ax.set()` で、題名（`title`）と縦軸の名前（`ylabel`）を付ける。
- `format_axis()` は目盛を見やすく整える。値は変えない。

最後の `fig.tight_layout()` は余白を整え、`plt.show()` で図を表示する。


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(rate_2024.index, rate_2024, linewidth=1)
ax.set(title="東京市場のドル・円（2024年）", ylabel="円／ドル")
format_axis(ax)
fig.tight_layout()
plt.show()


1月の143円台から円安が進み、7月初めには161円台まで上がった。その後は円高に戻って9月には140円台まで下がり、年末は157円台になっている。1年の間に20円ほどの幅で動いたことが分かる。


## 期間の代表値

`.mean()`、`.min()`、`.max()` で平均・最小・最大を求める。平均は観測のある各日を同じ重みで平均したもの。

In [ ]:
print(f"平均：{rate_2024.mean():.2f} 円／ドル")
print(f"最小：{rate_2024.min():.2f} 円／ドル")
print(f"最大：{rate_2024.max():.2f} 円／ドル")


最小や最大になった日付は、`.idxmin()` と `.idxmax()` で分かる。2024年は演習1で使うので、ここでは2023年のレートで試す。まず2023年の部分を `rate_2023` とし、最小になった日を調べる。


In [ ]:
rate_2023 = rate.loc["2023"]
rate_2023.idxmin()


`Timestamp('2023-01-16 00:00:00')` と表示された。`Timestamp` は日付と時刻を表す型で、ここでは2023年1月16日を指している。`.date()` を付けると日付だけになる。最大になった日と合わせて表示する。


In [ ]:
print(f"最小：{rate_2023.min():.2f} 円／ドル（{rate_2023.idxmin().date()}）")
print(f"最大：{rate_2023.max():.2f} 円／ドル（{rate_2023.idxmax().date()}）")


2023年は1月16日に最も円高（127.99円）、11月13日に最も円安（151.74円）だった。`.min()` と `.max()` は値そのものを、`.idxmin()` と `.idxmax()` はその値になった日付を返す。


## 円の予算と購入できるドル

たとえば15,000円で100ドル以上を買いたいとする。上限は 15,000 ÷ 100 ＝ 150円／ドルで、150円以下の日が条件に合う。

文字で書くと、予算 $B$ で $Q^*$ ドル以上を買えるのは、レートが上限以下の日である（$B,Q^*,S>0$）。

$$\frac{B}{S}\geq Q^* \quad\Longleftrightarrow\quad S\leq\frac{B}{Q^*}$$


## 条件に合う日の選び方

演習1では、条件に合う日だけを取り出す。そのときに使う `.loc` を、例で確かめておく。

`.loc[ ]` は、`[ ]` の中で指定した行を取り出す。第2回では日付を入れて期間を選んだ。


In [ ]:
rate_2024.loc["2024-07-01":"2024-07-05"]


`[ ]` の中には、日付の代わりに `True`／`False` を入れることもできる。まず、レートが160円以上かどうかを日ごとに調べる。比較の結果は、日付ごとの `True`／`False` になる。


In [ ]:
high = rate_2024 >= 160
high.loc["2024-06-24":"2024-07-12"]


6月27日から7月11日まで `True` が並んでいる。この期間はレートが160円以上だった。


この `high` を `.loc[ ]` に入れると、`True` の日だけが残る。


In [ ]:
rate_2024.loc[high]


160円以上だった日の日付とレートだけが並んだ。何日あったかは `len()` で数えられる。


In [ ]:
print(f"160円以上の日：{len(rate_2024.loc[high])} 日")


2行をまとめて `rate_2024.loc[rate_2024 >= 160]` と書いても同じ結果になる。演習1の追加問題では、`160` の代わりに予算から計算した上限を使い、`>=` を `<=` に変える。


## 演習1：12万円の両替予算

2024年の各観測日に、12万円を一度にドルへ換えるとする。

1. 最も多くのドルを買える日と最も少なくなる日を求め、それぞれの日付・為替レート・購入できるドルを示せ。
2. 800ドルを買うために、円／ドルのレートはいくら以下である必要があるか。予算と目標額から計算せよ。

追加問題：その条件に合う観測を選び、日数を数えよ。全245観測日のうち何%かも求め、この割合を「翌年に800ドルを買える確率」と呼べるか考えよ。

休日・欠測日のレートは補わない。

In [ ]:
budget_ex1 = 120_000
goal_usd_ex1 = 800
# rate_2024.min(), .max(), .idxmin(), .idxmax() を使う。
# 予算内で800ドルを買えるレートの上限を計算する。
# 条件に合う観測を .loc[...] で選び、len(...) で数える。


### 演習1の記入欄

- 最も多く買える日・レート・ドルの金額：
- 最も少なくなる日・レート・ドルの金額：
- 800ドルを買えるレートの上限：
- 追加問題：条件に合う日数・割合と、翌年について判断するために不足していること：


<details>
<summary>演習1のヒント</summary>

レートが低い日ほど多く買える。`budget_ex1 / rate_2024.min()` のように計算する。条件に合う観測は `limit_ex1 = budget_ex1 / goal_usd_ex1` として `rate_2024.loc[rate_2024 <= limit_ex1]` で選ぶ。割合を将来の確率とみなすには、将来の為替の動きについての仮定が要る。

</details>

## 出典

[日本銀行「外国為替市況」](https://www.boj.or.jp/statistics/market/forex/fxdaily/index.htm)の東京市場ドル・円スポット、17時時点（円／ドル）を使用。配布データの対象期間は2000〜2025年である。
